# Lab 1.4 — Wire Tina's memory

**Before you start:** select **Cell > Run All** to initialize the harness, then implement the `# ── YOUR WORK ──` cells.

Tina's harness includes a `MemoryStore` abstract class. You implement `remember()` and `recall()` backed by the `cortex-tina-memory` Elasticsearch index. The check runs a 3-turn conversation at temperature 0 — turn 3 must correctly resolve who "that" refers to.

**Acceptance:** Turn 3 returns the customer name and CTR threshold. `cortex-tina-memory` holds at least one document per turn.

In [ ]:
# ── Harness setup ────────────────────────────────────────────────────────────
import sys, os, json, pathlib, time
sys.path.insert(0, '/opt/ara/lib')

from tina.client import llm_client, es_client, model_fast
from tina.memory import MemoryStore

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
es     = es_client()
FAST   = model_fast()

RESULTS = pathlib.Path('/home/elastic/.traces')
RESULTS.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    'You are Tina, Cortex Bank and Trust\'s compliance assistant. '
    'Answer questions using your memory of the conversation and '
    'Cortex Bank policy. Be concise and cite policy IDs when relevant.'
)

print('Harness ready.')
print(f'  Model: {FAST}')
print(f'  Memory index: cortex-tina-memory')

In [ ]:
# ── The MemoryStore interface you will implement ───────────────────────────────
import inspect
print(inspect.getsource(MemoryStore))

---
## Your implementation

Implement `ElasticMemoryStore` below. Requirements:
- `remember(turn, content)`: index a document into `cortex-tina-memory` with the turn number and whatever entities and summary you decide are worth keeping.
- `recall(query, k=3)`: search `cortex-tina-memory` and return the k most relevant memories as dicts.

Hints:
- `es.index(index='cortex-tina-memory', document={...})` to write
- `es.search(index='cortex-tina-memory', body={'query': {'match': {'entities': query}}})` to read
- `content` must include at least `{"entities": "...", "summary": "..."}` for the check to find it

In [ ]:
# ── YOUR WORK ── Implement ElasticMemoryStore ─────────────────────────────────
from typing import Any

class ElasticMemoryStore(MemoryStore):
    def __init__(self, es_client, index='cortex-tina-memory'):
        self.es = es_client
        self.index = index

    def remember(self, turn: int, content: dict[str, Any]) -> None:
        # ── YOUR WORK ──
        # Index a document for this turn.
        # content must include 'entities' (str) and 'summary' (str).
        # Example:
        # self.es.index(index=self.index, document={'turn': turn, **content})
        pass

    def recall(self, query: str, k: int = 3) -> list[dict[str, Any]]:
        # ── YOUR WORK ──
        # Search cortex-tina-memory for memories relevant to query.
        # Return a list of document dicts, most relevant first.
        # Example:
        # resp = self.es.search(index=self.index,
        #     body={'query': {'match': {'entities': query}}}, size=k)
        # return [h['_source'] for h in resp['hits']['hits']]
        return []

memory = ElasticMemoryStore(es)
print('ElasticMemoryStore created. Implement remember() and recall() above.')

---
## Run the 3-turn conversation

After implementing `remember()` and `recall()`, run the cell below. It runs the three Cortex turns and uses your memory store between calls.

In [ ]:
# 3-turn conversation — run after implementing ElasticMemoryStore
TURNS = [
    'I am reviewing Elias Vance, account 4492. He made 10 daily ATM withdrawals of $9,500 each over two weeks.',
    'Are his recent wire transfers consistent with that withdrawal pattern?',
    'Is that reportable?',
]

memory.flush()  # start fresh
conversation_history = []
results = []

for turn_num, user_msg in enumerate(TURNS, 1):
    # Recall relevant memories
    memories = memory.recall(user_msg, k=3)
    memory_context = '\n'.join(
        f"[Memory turn {m.get('turn','')}]: {m.get('summary', str(m))}" for m in memories
    )

    system = SYSTEM_PROMPT
    if memory_context:
        system += f'\n\nConversation memory:\n{memory_context}'

    conversation_history.append({'role': 'user', 'content': user_msg})

    resp = client.chat.completions.create(
        model=FAST,
        messages=[{'role': 'system', 'content': system}] + conversation_history,
        temperature=0,
    )
    answer = resp.choices[0].message.content
    conversation_history.append({'role': 'assistant', 'content': answer})

    # Store memory for this turn — what should Tina remember?
    memory.remember(turn_num, {
        'entities': user_msg,   # you may want to be more selective
        'summary': f'Turn {turn_num}: {user_msg[:100]}',
    })

    results.append({'turn': turn_num, 'user': user_msg, 'tina': answer})
    print(f'Turn {turn_num}: {answer[:200]}')
    print()

(RESULTS / 'memory-conversation.json').write_text(json.dumps(results, indent=2))
print('Conversation recorded. Select Check in the sidebar.')